# 002 · Formatting and Parsing Dates

> **Standard Library › Dates & Times** · Python 3.14

Turning dates into text and back is where most date bugs live. Python uses C-style `%` directives (`strftime` / `strptime`) and ISO 8601 helpers (`isoformat` / `fromisoformat`) where .NET uses `ToString("yyyy-MM-dd")` and `ParseExact`.

**In this notebook you will learn:**

- `strftime` / `strptime` and how their `%` directives map to .NET custom format strings
- ISO 8601 round-tripping with `isoformat()` / `fromisoformat()` (the `"o"` format of Python)
- format specs in f-strings: `f"{dt:%Y-%m-%d}"`
- why `%A`, `%B` and `%c` depend on the locale
- converting to and from Unix timestamps

## .NET mental model

| .NET | Python |
|---|---|
| `dt.ToString("yyyy-MM-dd HH:mm")` | `dt.strftime("%Y-%m-%d %H:%M")` |
| `DateTime.ParseExact(s, "yyyy-MM-dd", CultureInfo.InvariantCulture)` | `datetime.strptime(s, "%Y-%m-%d")` |
| `DateTime.TryParseExact(...)` | `try: strptime(...) except ValueError:` |
| `dt.ToString("o")` (round-trip) | `dt.isoformat()` |
| `DateTime.Parse(s)` for ISO strings | `datetime.fromisoformat(s)` |
| `$"{dt:yyyy-MM-dd}"` | `f"{dt:%Y-%m-%d}"` |
| `DateTimeOffset.FromUnixTimeSeconds(n)` | `datetime.fromtimestamp(n, tz=timezone.utc)` |
| `dto.ToUnixTimeSeconds()` | `dt.timestamp()` |

There is no "parse anything" equivalent of `DateTime.Parse` in the stdlib: `strptime` needs an exact format, and `fromisoformat` only accepts ISO 8601.

## `strftime` — format a date as text

`strftime` ("string format time") takes a format string with `%` directives. Everything that is not a directive is copied literally — no need to quote or escape separators as you sometimes do in .NET.

```csharp
// C#
var dt = new DateTime(2026, 3, 5, 14, 30, 15, 123);
Console.WriteLine(dt.ToString("yyyy-MM-dd HH:mm:ss", CultureInfo.InvariantCulture));
Console.WriteLine(dt.ToString("dddd, dd MMMM yyyy", CultureInfo.InvariantCulture));
```

In [ ]:
from datetime import date, time, datetime, timedelta, timezone

dt = datetime(2026, 3, 5, 14, 30, 15, 123456)

print(dt.strftime("%Y-%m-%d %H:%M:%S"))       # 2026-03-05 14:30:15
print(dt.strftime("%A, %d %B %Y"))            # Thursday, 05 March 2026
print(dt.strftime("%d/%m/%y %I:%M %p"))       # 05/03/26 02:30 PM
print(dt.strftime("Week %V, day %j"))         # ISO week number, day of the year
print(dt.strftime("100%% done"))              # %% is a literal percent sign

## Directive cheat sheet

| Meaning | .NET | Python | Example |
|---|---|---|---|
| 4-digit year | `yyyy` | `%Y` | `2026` |
| 2-digit year | `yy` | `%y` | `26` |
| month, zero-padded | `MM` | `%m` | `03` |
| month abbreviation | `MMM` | `%b` | `Mar` |
| month name | `MMMM` | `%B` | `March` |
| day, zero-padded | `dd` | `%d` | `05` |
| weekday abbreviation | `ddd` | `%a` | `Thu` |
| weekday name | `dddd` | `%A` | `Thursday` |
| hour 00–23 | `HH` | `%H` | `14` |
| hour 01–12 | `hh` | `%I` | `02` |
| AM / PM | `tt` | `%p` | `PM` |
| minute | `mm` | `%M` | `30` |
| second | `ss` | `%S` | `15` |
| fraction | `fff` (ms) … `ffffff` (µs) | `%f` (always 6 digits, µs) | `123456` |
| UTC offset | `zzz` | `%z` / `%:z` | `+0100` / `+01:00` |
| time zone name | — | `%Z` | `UTC` |
| day of year | — | `%j` | `064` |
| ISO week | `ISOWeek.GetWeekOfYear` | `%V` (with `%G` ISO year) | `10` |
| literal `%` | — | `%%` | `%` |

⚠️ **Gotcha:** the letters mean different things. In .NET `MM` is month and `mm` is minute; in Python `%m` is **month** and `%M` is **minute**. Mixing them up produces valid-looking but wrong output, with no error.

In [ ]:
print(dt.strftime("%Y-%m-%d"))   # correct: month
print(dt.strftime("%Y-%M-%d"))   # oops: MINUTE in the month position → 2026-30-05

`%f` always gives 6 digits of microseconds. For .NET-style milliseconds (`fff`), slice the string or use `isoformat(timespec="milliseconds")`:

In [ ]:
print(dt.strftime("%H:%M:%S.%f"))            # 14:30:15.123456
print(dt.strftime("%H:%M:%S.%f")[:-3])       # 14:30:15.123  (like HH:mm:ss.fff)
print(dt.isoformat(timespec="milliseconds")) # 2026-03-05T14:30:15.123

⚠️ **Gotcha:** there is no portable directive for a *non-padded* day or month (.NET's `d` / `M`). `%-d` works on Linux and macOS, `%#d` on Windows, and neither works everywhere. Build the string yourself with the attributes instead:

In [ ]:
print(f"{dt.day}/{dt.month}/{dt.year}")               # 5/3/2026 — portable
print(f"{dt:%B} {dt.day}, {dt.year}")                 # March 5, 2026

## Format specs in f-strings

Dates implement `__format__`, so any `strftime` pattern can go after the colon in an f-string — just like `$"{dt:yyyy-MM-dd}"` in C#. The built-in `format()` function does the same thing. *(see Syntax › Data Types › 003 · String Formatting)*

```csharp
// C#
Console.WriteLine($"Order shipped on {dt:dd MMM yyyy} at {dt:HH:mm}");
```

In [ ]:
print(f"Order shipped on {dt:%d %b %Y} at {dt:%H:%M}")
print(format(dt.date(), "%A"))              # Thursday
print(f"{dt}")                              # no spec → str(dt), the ISO-like default
print(f"{dt=:%Y-%m-%d}")                    # self-documenting = works too

## `strptime` — parse text with an exact format

`datetime.strptime(text, format)` is Python's `DateTime.ParseExact`. It uses the same directives as `strftime` and raises `ValueError` if the text does not match **exactly** — there is no `TryParseExact`, you catch the exception instead.

```csharp
// C#
var dt = DateTime.ParseExact("05.03.2026 14:30", "dd.MM.yyyy HH:mm", CultureInfo.InvariantCulture);
bool ok = DateTime.TryParseExact("bad", "dd.MM.yyyy", CultureInfo.InvariantCulture,
                                 DateTimeStyles.None, out var parsed);
```

In [ ]:
parsed = datetime.strptime("05.03.2026 14:30", "%d.%m.%Y %H:%M")
print(repr(parsed))

print(datetime.strptime("Thu, 05 Mar 2026", "%a, %d %b %Y"))
print(datetime.strptime("2026-03-05 14:30:15.12", "%Y-%m-%d %H:%M:%S.%f"))  # %f accepts 1–6 digits

Parse failures give a fairly helpful message. Extra trailing text is an error too:

In [ ]:
for text in ["2026-03-05", "05/03/2026", "2026-03-05 extra", "2026-02-30"]:
    try:
        print(datetime.strptime(text, "%Y-%m-%d").date())
    except ValueError as e:
        print(f"{text!r}: {type(e).__name__}: {e}")

A small `try_parse` helper gives you `TryParseExact`-style behaviour by returning `None` on failure:

In [ ]:
def try_parse(text: str, fmt: str) -> datetime | None:
    try:
        return datetime.strptime(text, fmt)
    except ValueError:
        return None

print(try_parse("2026-03-05", "%Y-%m-%d"))
print(try_parse("not a date", "%Y-%m-%d"))

`strptime` is a class method of `datetime`, so it always returns a `datetime`. Python 3.14 added `date.strptime` and `time.strptime` for when you only want one part:

In [ ]:
print(repr(date.strptime("05.03.2026", "%d.%m.%Y")))     # new in 3.14
print(repr(time.strptime("14:30", "%H:%M")))             # new in 3.14 (datetime.time, not the time module!)

# before 3.14 you would write:
print(repr(datetime.strptime("05.03.2026", "%d.%m.%Y").date()))

Parsing a UTC offset with `%z` gives you an **aware** datetime. `%z` accepts `+0100`, `+01:00` and `Z`:

In [ ]:
for text in ["2026-03-05T14:30:15+0100", "2026-03-05T14:30:15+01:00", "2026-03-05T14:30:15Z"]:
    parsed = datetime.strptime(text, "%Y-%m-%dT%H:%M:%S%z")
    print(parsed, "| offset:", parsed.utcoffset())

## ISO 8601: `isoformat` and `fromisoformat`

For machine-readable text (JSON, logs, databases, APIs) use ISO 8601. `isoformat()` is the equivalent of .NET's round-trip `"o"` format, and `fromisoformat()` parses it back.

```csharp
// C#
string s = dto.ToString("o");                // 2026-03-05T14:30:15.1234560+01:00
var back = DateTimeOffset.Parse(s, CultureInfo.InvariantCulture);
```

In [ ]:
print(dt.isoformat())                                   # 2026-03-05T14:30:15.123456
print(dt.isoformat(sep=" ", timespec="seconds"))        # 2026-03-05 14:30:15
print(dt.date().isoformat(), dt.time().isoformat("minutes"))

aware = datetime(2026, 3, 5, 14, 30, tzinfo=timezone(timedelta(hours=1)))
print(aware.isoformat())                                # offset included: +01:00

`str(dt)` is the same as `dt.isoformat(sep=" ")`. Round-tripping is lossless:

In [ ]:
text = aware.isoformat()
back = datetime.fromisoformat(text)
print(back == aware, back.utcoffset())     # True 1:00:00

Since Python 3.11, `fromisoformat` accepts most ISO 8601 forms, including a trailing `Z`, the compact basic format and ISO week dates. It still rejects anything that is not ISO — like `05/03/2026`:

In [ ]:
for text in ["2026-03-05", "2026-03-05T14:30:15Z", "20260305T143015", "2026-W10-4", "05/03/2026"]:
    try:
        print(f"{text!r:24} → {datetime.fromisoformat(text)!r}")
    except ValueError as e:
        print(f"{text!r:24} → {type(e).__name__}: {e}")

**Tip:** store and exchange dates as ISO 8601 strings in UTC (`2026-03-05T13:30:00+00:00`). They sort correctly as plain text, are unambiguous across cultures, and every language parses them.

## Locale dependence

`%a`, `%A`, `%b`, `%B`, `%p`, `%c`, `%x` and `%X` produce **locale-dependent** text — they are the `CultureInfo.CurrentCulture` part of Python. By default Python runs in the neutral `"C"` locale, so you get English names; but any code that calls `locale.setlocale` changes the output process-wide.

```csharp
// C# — culture is explicit, per call
dt.ToString("dddd", new CultureInfo("de-DE"));      // Donnerstag
dt.ToString("dddd", CultureInfo.InvariantCulture);  // Thursday
```

In [ ]:
import locale

print(dt.strftime("%A %d %B | %c"))      # "C" locale: English names

saved = locale.setlocale(locale.LC_TIME)        # query the current setting
try:
    locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")   # name is OS-specific!
    print(dt.strftime("%A %d %B"))       # e.g. Donnerstag 05 März
except locale.Error as e:
    print(f"{type(e).__name__}: {e} (locale not installed on this machine)")
finally:
    locale.setlocale(locale.LC_TIME, saved)     # always restore global state

print(dt.strftime("%A"))                 # back to Thursday

⚠️ **Gotcha:** unlike .NET, you cannot pass a culture to a single `strftime` call — the locale is **global, process-wide state**, and locale names differ between operating systems. For machine-readable output avoid the name directives entirely (use ISO 8601). For user-facing localised dates, the third-party `Babel` package is the usual choice.

⚠️ **Gotcha:** `strptime` is locale-dependent too: `%b` only matches month abbreviations of the *current* locale.

## Unix timestamps

A Unix timestamp is the number of seconds since 1970-01-01 00:00 UTC, as a `float` in Python.

```csharp
// C#
var dto = DateTimeOffset.FromUnixTimeSeconds(1772721015);
long seconds = dto.ToUnixTimeSeconds();
long millis = dto.ToUnixTimeMilliseconds();
```

In [ ]:
ts = 1772721015
from_ts = datetime.fromtimestamp(ts, tz=timezone.utc)   # C#: DateTimeOffset.FromUnixTimeSeconds
print(from_ts)                                          # 2026-03-05 14:30:15+00:00

print(from_ts.timestamp())                              # 1772721015.0 (C#: ToUnixTimeSeconds)
print(int(from_ts.timestamp() * 1000))                  # milliseconds (C#: ToUnixTimeMilliseconds)
print(datetime.fromtimestamp(1772721015123 / 1000, tz=timezone.utc))   # from milliseconds

⚠️ **Gotcha:** without `tz=`, `fromtimestamp` returns a **naive local time**, and `.timestamp()` on a naive datetime assumes it is **local time** — so results depend on the machine's time zone. Always pass `tz=timezone.utc` and work with aware datetimes when converting timestamps.

In [ ]:
naive = datetime(2026, 3, 5, 14, 30, 15)            # no time zone attached
aware_utc = naive.replace(tzinfo=timezone.utc)      # explicitly UTC

print(aware_utc.timestamp())                        # always 1772721015.0
# naive is read as LOCAL time, so the gap is this machine's UTC offset on that date
print((aware_utc.timestamp() - naive.timestamp()) / 3600, "hours")   # 0.0 only where local time == UTC

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Format | `dt.ToString("yyyy-MM-dd HH:mm")` | `dt.strftime("%Y-%m-%d %H:%M")` |
| Format inline | `$"{dt:yyyy-MM-dd}"` | `f"{dt:%Y-%m-%d}"` |
| Parse exact | `DateTime.ParseExact(s, fmt, ...)` | `datetime.strptime(s, fmt)` |
| Try-parse | `DateTime.TryParseExact(...)` | `try` / `except ValueError` |
| Date-only parse | `DateOnly.ParseExact(...)` | `date.strptime(...)` (3.14+) |
| Round-trip string | `dt.ToString("o")` | `dt.isoformat()` |
| Parse ISO 8601 | `DateTime.Parse(s)` | `datetime.fromisoformat(s)` |
| Milliseconds | `"fff"` | `isoformat(timespec="milliseconds")` |
| From Unix seconds | `DateTimeOffset.FromUnixTimeSeconds(n)` | `datetime.fromtimestamp(n, tz=timezone.utc)` |
| To Unix seconds | `dto.ToUnixTimeSeconds()` | `dt.timestamp()` |
| Culture-specific names | `ToString("dddd", culture)` | global `locale.setlocale` (or Babel) |

## Exercises

1. Parse `"2026-03-05 14:30"` and print it as `"Thursday 5 March 2026, 2:30 PM"` (no leading zeros on the day or the hour, portable code).
2. Write `parse_any(text) -> date | None` that tries the formats `"%Y-%m-%d"`, `"%d/%m/%Y"` and `"%d %b %Y"` in turn and returns the first successful parse (or `None`). Test it with `"2026-03-05"`, `"05/03/2026"`, `"5 Mar 2026"` and `"March"`.
3. Convert the Unix timestamp in milliseconds `1772721015500` to an aware UTC datetime and print it in ISO 8601 with millisecond precision.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from datetime import date, datetime, timezone

# 1 — mix strftime for names with attributes for the unpadded numbers
dt = datetime.strptime("2026-03-05 14:30", "%Y-%m-%d %H:%M")
hour12 = dt.hour % 12 or 12
print(f"{dt:%A} {dt.day} {dt:%B %Y}, {hour12}:{dt:%M %p}")   # Thursday 5 March 2026, 2:30 PM

# 2 — loop over the candidate formats, return on the first success
def parse_any(text: str) -> date | None:
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"):
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue
    return None

for text in ["2026-03-05", "05/03/2026", "5 Mar 2026", "March"]:
    print(text, "→", parse_any(text))

# 3 — divide by 1000 and always pass tz=
moment = datetime.fromtimestamp(1772721015500 / 1000, tz=timezone.utc)
print(moment.isoformat(timespec="milliseconds"))   # 2026-03-05T14:30:15.500+00:00
```

</details>